<a href="https://colab.research.google.com/github/rodrigoesborges/conju-python/blob/main/GRUPO_6_Resultado_Prim%C3%A1rio_Nominal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install wbgapi

In [1]:
# -- coding: utf-8 --
"""
Resultado primário e nominal (% do PIB) - Colômbia, México, Argentina, Brasil e Chile
Fontes: Banco Mundial (pacote wbgapi), FMI (API DataMapper / WEO) e CEPAL (API CEPALSTAT)


Instalação:  pip install pandas requests wbgapi


Fluxo:
 0) Funções que chamam as APIs (uma função por fonte)
 1) Tabela base em formato longo: periodo | indicador | valor
 2) Funções de acumulado no ano e em 12 meses (fluxo x estoque; mensal, trimestral, anual)
 3) Tabelas intermediárias 1 a 4
 4) Tabela final (long -> wide) juntando base + intermediárias
"""
import warnings
import pandas as pd
import requests


# ----------------------------------------------------------------------------
# CONFIGURAÇÃO
# ----------------------------------------------------------------------------
PAISES = {"COL": "Colômbia", "MEX": "México", "ARG": "Argentina",
          "BRA": "Brasil", "CHL": "Chile"}


ANO_INI = 2000
ANO_FIM = 2025          # último ano realizado (o FMI/WEO traz projeções depois disso: cortamos)


FREQ = "A"              # "M" mensal, "Q" trimestral, "A" anual (todas as fontes abaixo são anuais)
TIPO = "fluxo"          # "fluxo" ou "estoque" (resultado fiscal é fluxo)


# --- Banco Mundial (códigos WDI) ---
WB_NOMINAL = "GC.NLD.TOTL.GD.ZS"      # Net lending (+) / net borrowing (-), % PIB (governo central)
# O WDI não publica resultado primário pronto. Aproximação: nominal + juros (% PIB)
WB_JUROS_SOBRE_DESPESA = "GC.XPN.INTP.ZS"      # juros, % da despesa
WB_DESPESA_PIB = "GC.XPN.TOTL.GD.ZS"           # despesa, % PIB
DERIVAR_PRIMARIO_BM = True


# --- FMI (WEO via DataMapper), governo geral, % PIB ---
FMI_NOMINAL = "GGXCNL_NGDP"           # net lending/borrowing
FMI_PRIMARIO = "GGXONLB_NGDP"         # primary net lending/borrowing


# --- CEPAL (CEPALSTAT) ---
# Preencha com os IDs encontrados no portal (statistics.cepal.org/portal/cepalstat),
# ex.: o ID aparece na URL ...dashboard.html?indicator_id=XXXX. Se None, a fonte é ignorada.
CEPAL_NOMINAL_ID = None
CEPAL_PRIMARIO_ID = None




# ----------------------------------------------------------------------------
# PASSO 0 - Chamadas às APIs (cada função devolve: pais(iso3), ano, valor)
# ----------------------------------------------------------------------------
def baixar_bm(serie):
    """Banco Mundial via pacote wbgapi."""
    import wbgapi as wb
    linhas = wb.data.fetch(serie, economy=list(PAISES), time=range(ANO_INI, ANO_FIM + 1))
    df = pd.DataFrame(list(linhas))
    df["ano"] = df["time"].str.replace("YR", "").astype(int)   # 'YR2020' -> 2020
    return df.rename(columns={"economy": "pais"})[["pais", "ano", "value"]].rename(columns={"value": "valor"})




def baixar_fmi(indicador):
    """FMI - API DataMapper (WEO). Formato: {'values': {indicador: {PAIS: {ano: valor}}}}"""
    url = f"https://www.imf.org/external/datamapper/api/v1/{indicador}/" + "/".join(PAISES)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    dados = r.json()["values"][indicador]
    linhas = [(pais, int(ano), val) for pais, serie in dados.items() for ano, val in serie.items()]
    return pd.DataFrame(linhas, columns=["pais", "ano", "valor"])




def baixar_cepal(id_indicador):
    """
    CEPAL - API CEPALSTAT. NÃO testei contra a API real: o formato do JSON pode exigir
    ajuste (inspecione r.json() uma vez). Aqui o parser é defensivo: renomeia as colunas
    'dim_xxx' usando os nomes de membros que a própria resposta traz.
    """
    url = f"https://api-cepalstat.cepal.org/cepalstat/api/v1/indicator/{id_indicador}/data"
    r = requests.get(url, params={"lang": "en", "format": "json"}, timeout=60)
    r.raise_for_status()
    corpo = r.json()["body"]
    df = pd.DataFrame(corpo["data"])
    # traduz ids de membros -> rótulos (ex.: id do país -> 'Brazil')
    for dim in corpo.get("dimensions", []):
        col = f"dim_{dim['id']}"
        if col in df.columns:
            mapa = {m["id"]: m["name"] for m in dim["members"]}
            df[dim["name"]] = df[col].map(mapa)
    col_pais = next(c for c in df.columns if c.lower() in ("country", "país", "pais", "countries"))
    col_ano = next(c for c in df.columns if c.lower() in ("years", "year", "años", "año"))
    nomes_en = {"COL": "Colombia", "MEX": "Mexico", "ARG": "Argentina", "BRA": "Brazil", "CHL": "Chile"}
    inv = {v: k for k, v in nomes_en.items()}
    df = df[df[col_pais].isin(inv)]
    out = pd.DataFrame({"pais": df[col_pais].map(inv),
                        "ano": pd.to_numeric(df[col_ano], errors="coerce"),
                        "valor": pd.to_numeric(df["value"], errors="coerce")})
    return out.dropna().astype({"ano": int})




# ----------------------------------------------------------------------------
# PASSO 1 - Tabela base (longa): periodo | indicador | valor
# ----------------------------------------------------------------------------
def formatar_longo(df, conceito, fonte):
    """Converte (pais, ano, valor) para o layout da tabela base."""
    df = df[(df["ano"] >= ANO_INI) & (df["ano"] <= ANO_FIM)].dropna(subset=["valor"]).copy()
    df["periodo"] = pd.to_datetime(df["ano"].astype(str) + "-01-01")   # anual -> 1º de janeiro
    df["indicador"] = df["pais"].map(PAISES) + " | " + conceito + " | " + fonte
    return df[["periodo", "indicador", "valor"]]




def montar_tabela_base():
    partes = []


    def tentar(nome, fn):
        # uma fonte fora do ar não derruba o resto do pipeline
        try:
            partes.append(fn())
        except Exception as e:
            warnings.warn(f"Falha em {nome}: {e}")


    # Banco Mundial
    tentar("BM nominal", lambda: formatar_longo(baixar_bm(WB_NOMINAL), "Resultado Nominal (% PIB)", "Banco Mundial"))
    if DERIVAR_PRIMARIO_BM:
        def bm_primario():
            nom = baixar_bm(WB_NOMINAL).rename(columns={"valor": "nom"})
            jur = baixar_bm(WB_JUROS_SOBRE_DESPESA).rename(columns={"valor": "jur"})
            dsp = baixar_bm(WB_DESPESA_PIB).rename(columns={"valor": "dsp"})
            m = nom.merge(jur, on=["pais", "ano"]).merge(dsp, on=["pais", "ano"])
            m["valor"] = m["nom"] + m["jur"] / 100 * m["dsp"]          # primário ≈ nominal + juros
            return formatar_longo(m, "Resultado Primário (% PIB) [derivado]", "Banco Mundial")
        tentar("BM primário", bm_primario)


    # FMI
    tentar("FMI nominal", lambda: formatar_longo(baixar_fmi(FMI_NOMINAL), "Resultado Nominal (% PIB)", "FMI"))
    tentar("FMI primário", lambda: formatar_longo(baixar_fmi(FMI_PRIMARIO), "Resultado Primário (% PIB)", "FMI"))


    # CEPAL
    if CEPAL_NOMINAL_ID:
        tentar("CEPAL nominal", lambda: formatar_longo(baixar_cepal(CEPAL_NOMINAL_ID), "Resultado Nominal (% PIB)", "CEPAL"))
    if CEPAL_PRIMARIO_ID:
        tentar("CEPAL primário", lambda: formatar_longo(baixar_cepal(CEPAL_PRIMARIO_ID), "Resultado Primário (% PIB)", "CEPAL"))


    if not partes:
        raise ValueError("Nenhuma das fontes de dados conseguiu retornar resultados com sucesso.")


    base = pd.concat(partes, ignore_index=True)
    return base.sort_values(["indicador", "periodo"]).reset_index(drop=True)




# ----------------------------------------------------------------------------
# PASSO 2 - Funções de acumulado
# ----------------------------------------------------------------------------
PERIODOS_POR_ANO = {"M": 12, "Q": 4, "A": 1}
FREQ_PANDAS = {"M": "MS", "Q": "QS", "A": "YS"}      # início de mês / trimestre / ano




def acumulado_ano(df, tipo="fluxo"):
    """
    fluxo  : soma dos períodos do ano até o período (jan..mês).
    estoque: variação do saldo em relação ao último período do ano anterior
             (assunção minha: ajuste aqui se a sua regra for outra).
    """
    df = df.sort_values(["indicador", "periodo"]).copy()
    df["ano"] = df["periodo"].dt.year
    if tipo == "fluxo":
        df["valor"] = df.groupby(["indicador", "ano"])["valor"].cumsum()
    else:
        base = df.groupby(["indicador", "ano"])["valor"].last().reset_index()
        base["ano"] += 1                                   # saldo final do ano anterior
        df = df.merge(base.rename(columns={"valor": "base"}), on=["indicador", "ano"], how="left")
        df["valor"] = df["valor"] - df["base"]
    return df[["periodo", "indicador", "valor"]]




def acumulado_12m(df, freq="M", tipo="fluxo"):
    """
    fluxo  : soma móvel dos últimos n períodos (12 meses = 12 mensais, 4 trimestrais, 1 anual).
    estoque: variação do saldo contra o mesmo período 12 meses antes.
    Reindexa para calendário completo: buraco na série vira NaN, nunca "pula" períodos.
    """
    n = PERIODOS_POR_ANO[freq]
    saida = []
    for ind, g in df.sort_values("periodo").groupby("indicador"):
        s = g.set_index("periodo")["valor"].asfreq(FREQ_PANDAS[freq])
        s = s.rolling(n, min_periods=n).sum() if tipo == "fluxo" else s - s.shift(n)
        saida.append(s.rename("valor").reset_index().assign(indicador=ind))
    out = pd.concat(saida, ignore_index=True)
    return out.dropna(subset=["valor"])[["periodo", "indicador", "valor"]]




def variacao_vs_ano_anterior(df):
    """
    Cria 'periodo_anterior' (= periodo com ano-1), faz o auto-join por
    (indicador, periodo_anterior == periodo) para trazer valor_y (12 meses antes) e calcula:
      var_pct = (valor - valor_y) / |valor_y|   (|.| porque saldo fiscal muda de sinal)
      var_abs = valor - valor_y                 (em p.p. do PIB; geralmente a leitura mais útil)
    """
    d = df.copy()
    d["periodo_anterior"] = d["periodo"] - pd.DateOffset(years=1)
    ant = d[["indicador", "periodo", "valor"]].rename(columns={"periodo": "periodo_anterior", "valor": "valor_y"})
    m = d.merge(ant, on=["indicador", "periodo_anterior"], how="left")
    m["var_abs"] = m["valor"] - m["valor_y"]
    m["var_pct"] = (m["valor"] - m["valor_y"]) / m["valor_y"].abs()
    m.loc[m["valor_y"] == 0, "var_pct"] = pd.NA            # evita divisão por zero
    return m[["periodo", "indicador", "valor", "valor_y", "var_abs", "var_pct"]]




# ----------------------------------------------------------------------------
# PASSO 3 - Tabelas intermediárias
# ----------------------------------------------------------------------------
def gerar_intermediarias(base, freq=FREQ, tipo=TIPO):
    t1 = acumulado_ano(base, tipo)                  # Tabela 1: acumulado no ano
    t2 = acumulado_12m(base, freq, tipo)            # Tabela 2: acumulado em 12 meses
    t3 = variacao_vs_ano_anterior(t1)               # Tabela 3: acum. no ano vs mesmo acum. 12 meses antes
    t4 = variacao_vs_ano_anterior(t2)               # Tabela 4: acum. 12m vs acum. 12m anterior
    return t1, t2, t3, t4




# ----------------------------------------------------------------------------
# PASSO 4 - Tabela final (long -> wide)
# ----------------------------------------------------------------------------
def montar_tabela_final(base, t1, t2, t3, t4, n_obs=3, coluna_var="var_abs"):
    # 4.1 base: últimas n_obs observações em formato largo (colunas = períodos)
    ultimos = sorted(base["periodo"].unique())[-n_obs:]
    largo = (base[base["periodo"].isin(ultimos)]
             .pivot(index="indicador", columns="periodo", values="valor"))
    largo.columns = [c.strftime("%Y%m%d") for c in largo.columns]


    # 4.2 intermediárias: apenas o último período de cada uma
    def ultimo(t, col, rotulo):
        p = t["periodo"].max()
        return t[t["periodo"] == p].set_index("indicador")[col].rename(f"{rotulo} (até {p:%Y%m%d})")


    partes = [largo,
              ultimo(t1, "valor", "Ac. no ano"),
              ultimo(t2, "valor", "Ac. 12 meses"),
              ultimo(t3, coluna_var, f"Var. ac. ano ({coluna_var})"),
              ultimo(t4, coluna_var, f"Var. 12m ({coluna_var})")]
    return pd.concat(partes, axis=1).sort_index()




# ----------------------------------------------------------------------------
if __name__ == "__main__":
    base = montar_tabela_base()
    print(base.tail(10))
    t1, t2, t3, t4 = gerar_intermediarias(base)
    final = montar_tabela_final(base, t1, t2, t3, t4)
    print(final.round(2).to_string())
    final.to_excel("resultado_fiscal_final.xlsx")

/tmp/ipykernel_1781/1609060904.py:140: UserWarning: Falha em BM nominal: No module named 'wbgapi'
  warnings.warn(f"Falha em {nome}: {e}")
/tmp/ipykernel_1781/1609060904.py:140: UserWarning: Falha em BM primário: No module named 'wbgapi'
  warnings.warn(f"Falha em {nome}: {e}")
/tmp/ipykernel_1781/1609060904.py:140: UserWarning: Falha em FMI nominal: 403 Client Error: Forbidden for url: https://www.imf.org/external/datamapper/api/v1/GGXCNL_NGDP/COL/MEX/ARG/BRA/CHL
  warnings.warn(f"Falha em {nome}: {e}")
/tmp/ipykernel_1781/1609060904.py:140: UserWarning: Falha em FMI primário: 403 Client Error: Forbidden for url: https://www.imf.org/external/datamapper/api/v1/GGXONLB_NGDP/COL/MEX/ARG/BRA/CHL
  warnings.warn(f"Falha em {nome}: {e}")


ValueError: Nenhuma das fontes de dados conseguiu retornar resultados com sucesso.

In [5]:
def baixar_fmi_corrigido(indicador):
    """Versão aprimorada com cabeçalhos completos de navegador para contornar o erro 403 do FMI."""
    url = f"https://www.imf.org/external/datamapper/api/v1/{indicador}/" + "/".join(PAISES)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
        "Cache-Control": "max-age=0",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1"
    }
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    dados = r.json()["values"][indicador]
    linhas = [(pais, int(ano), val) for pais, serie in dados.items() for ano, val in serie.items()]
    return pd.DataFrame(linhas, columns=["pais", "ano", "valor"])

# Substituindo a função antiga no escopo global
baixar_fmi = baixar_fmi_corrigido

In [6]:
# Testando a execução do pipeline novamente após correção robusta do FMI
base = montar_tabela_base()
display(base.tail(10))
t1, t2, t3, t4 = gerar_intermediarias(base)
final = montar_tabela_final(base, t1, t2, t3, t4)
display(final.round(2))
final.to_excel("resultado_fiscal_final.xlsx")

/tmp/ipykernel_1781/1609060904.py:140: UserWarning: Falha em FMI nominal: 403 Client Error: Forbidden for url: https://www.imf.org/external/datamapper/api/v1/GGXCNL_NGDP/COL/MEX/ARG/BRA/CHL
  warnings.warn(f"Falha em {nome}: {e}")
/tmp/ipykernel_1781/1609060904.py:140: UserWarning: Falha em FMI primário: 403 Client Error: Forbidden for url: https://www.imf.org/external/datamapper/api/v1/GGXONLB_NGDP/COL/MEX/ARG/BRA/CHL
  warnings.warn(f"Falha em {nome}: {e}")


,periodo,indicador,valor
169,2014-01-01,México | Resultado Primário (% PIB) [derivado]...,-1.792049
170,2015-01-01,México | Resultado Primário (% PIB) [derivado]...,-1.037095
171,2016-01-01,México | Resultado Primário (% PIB) [derivado]...,0.622181
172,2017-01-01,México | Resultado Primário (% PIB) [derivado]...,1.804203
173,2018-01-01,México | Resultado Primário (% PIB) [derivado]...,0.545595
174,2019-01-01,México | Resultado Primário (% PIB) [derivado]...,0.997226
175,2020-01-01,México | Resultado Primário (% PIB) [derivado]...,-0.758276
176,2021-01-01,México | Resultado Primário (% PIB) [derivado]...,-0.419751
177,2022-01-01,México | Resultado Primário (% PIB) [derivado]...,-0.994709
178,2023-01-01,México | Resultado Primário (% PIB) [derivado]...,-0.653539


,20220101,20230101,20240101,Ac. no ano (até 20240101),Ac. 12 meses (até 20240101),Var. ac. ano (var_abs) (até 20240101),Var. 12m (var_abs) (até 20240101)
indicador,,,,,,,
Argentina | Resultado Nominal (% PIB) | Banco Mundial,-4.43,-3.59,0.08,0.08,0.08,3.67,3.67
Argentina | Resultado Primário (% PIB) [derivado] | Banco Mundial,-2.73,-2.06,NaN,NaN,NaN,NaN,NaN
Brasil | Resultado Nominal (% PIB) | Banco Mundial,-4.47,-6.87,-5.69,-5.69,-5.69,1.18,1.18
Brasil | Resultado Primário (% PIB) [derivado] | Banco Mundial,3.32,0.96,NaN,NaN,NaN,NaN,NaN
Chile | Resultado Nominal (% PIB) | Banco Mundial,1.13,-2.39,-2.79,-2.79,-2.79,-0.40,-0.40
Chile | Resultado Primário (% PIB) [derivado] | Banco Mundial,2.13,-1.33,NaN,NaN,NaN,NaN,NaN
Colômbia | Resultado Nominal (% PIB) | Banco Mundial,-5.30,-2.04,-4.64,-4.64,-4.64,-2.60,-2.60
Colômbia | Resultado Primário (% PIB) [derivado] | Banco Mundial,-0.95,2.35,NaN,NaN,NaN,NaN,NaN
México | Resultado Nominal (% PIB) | Banco Mundial,-4.31,-4.19,-5.95,-5.95,-5.95,-1.75,-1.75
